# Ossia Demo Notebook

End-to-end walkthrough of the Ossia support agent.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from ossia.agent import build_agent_async
from ossia.config import Settings

settings = Settings()
print(settings.provider, settings.model)

## Build the agent (async context manager keeps MCP sessions alive)

In [ ]:
from contextlib import AsyncExitStack

_stack = AsyncExitStack()
await _stack.__aenter__()
agent = await _stack.enter_async_context(
    build_agent_async(settings=settings, include_mcp_tools=True)
)
agent

## Run a simple query

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "demo-001"}}

result = await agent.ainvoke(
    {"messages": [HumanMessage(content="What are Nebius Serverless Jobs?")]},
    config,
)
for msg in result["messages"]:
    print(msg.type, ":", str(msg.content)[:200])

## Stream events for UI feedback

In [ ]:
async for event in agent.astream_events(
    {"messages": [HumanMessage(content="How do I reset my endpoint credentials?")]},
    {"configurable": {"thread_id": "demo-002"}},
    version="v2",
):
    kind = event["event"]
    if kind in {"on_chat_model_stream", "on_tool_start", "on_tool_end"}:
        print(kind, event.get("name"), str(event.get("data", ""))[:200])

## Clean up MCP sessions when done

In [ ]:
await _stack.aclose()